# 05 · Persiapan Data Meteorologi — Bab 6

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 6: memuat data, QC, eksplorasi, *feature engineering*, normalisasi, dan split berbasis waktu. Data nyata terbuka (hasil `scripts/download_*.py`: GHCN-Daily/CHIRPS + ERA5/ERA5-Land + indeks iklim) dipakai langsung; tanpa data, notebook berhenti dengan instruksi unduh.

## 1. Setup & Data Contoh

Notebook wajib memakai **data nyata terbuka** (ERA5-Land jakarta + CHIRPS + indeks iklim, via `scripts/download_*.py`). Data sintetik tidak lagi dipakai.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

np.random.seed(42)

# ---------------------------------------------------------------------------
# Data WAJIB nyata dari repoti buku: ERA5-Land + CHIRPS + indeks iklim
# (hasilkan via scripts/download_*.py ke manuscripts/ch-08/09/data).
# Jika belum tersedia: FileNotFoundError dengan instruksi unduh.
# ---------------------------------------------------------------------------
def _buscar_raiz():
    """Cari pasta repo buku yang memuat manuscripts/ (portabel Colab+local)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "manuscripts").exists():
            return p
    return Path.cwd()

_BASE = _buscar_raiz() / "manuscripts/ch-09-studi-kasus-curah-hujan-terbuka/data"

def cargar_nyata():
    """Ambil data nyata terbuka (ERA5-Land jakarta + CHIRPS) bila ada."""
    era_files = sorted((_BASE / "era5" / "jakarta").glob("era5land_jakarta_*_daily.csv"))
    if not era_files:
        return None
    era = pd.concat([pd.read_csv(p, parse_dates=["tanggal"]).set_index("tanggal")
                     for p in era_files]).sort_index()
    era = era[~era.index.duplicated(keep="first")]
    df = pd.DataFrame({
        "r_hujan": era["tp_mm"],                       # target: hujan grid ERA5-Land (mm/hari)
        "suhu": era["t2m_c"],                          # suhu 2 m (degC)
        "angin": (era["u10"] ** 2 + era["v10"] ** 2) ** 0.5,   # speed angin (m/s)
    })
    return df

df = cargar_nyata()
if df is None:
    raise FileNotFoundError(
        "Data nyata ERA5-Land jakarta belum tersedia. Jalankan:\n"
        "  python scripts/download_era5.py --stations jakarta "
        "--years 2010-2025 --land --process"
    )
ETIQUETA = "data nyata terbuka: ERA5-Land (jakarta) + indeks iklim nyata"

print("Sumber:", ETIQUETA)
print(df.head())
print(df.describe())

## 2. Quality Control & Imputasi

In [2]:
print("Nilai hilang per kolom:")
print(df.isna().sum())

# imputasi sederhana: hujan gap -> 0 (konservativ), kolom kontinu -> ffill/mean
df_clean = df.copy()
for c in df_clean.columns:
    if c == "r_hujan":
        df_clean[c] = df_clean[c].fillna(0.0)
    elif c != "chirps_mm":          # kolom verifikasi dipertahakan as-is
        df_clean[c] = df_clean[c].ffill().fillna(df_clean[c].mean())
print("Sisa hilang:", int(df_clean.isna().sum().sum()))

Nilai hilang per kolom:
r_hujan    0
suhu       0
angin      0
dtype: int64
Sisa hilang: 0


## 3. Eksplorasi Distribusi (Gambar 6.1)

In [3]:
plt.figure(figsize=(6.5,4))
df_clean["r_hujan"].hist(bins=60, color="#4a90e2", edgecolor="white")
plt.xlabel("Curah hujan harian (mm)"); plt.ylabel("Frekuensi")
plt.title("Distribusi curah hujan harian (contoh)")
plt.tight_layout(); plt.show()

## 3b. Korelasi Silang (Tujuan Bab 6 #3)

Selang korelasi Pearson antar fitur — petunjuk fitur redundan (untuk Bab 7/9:
fitur yang tinggi berkorelasi duplikasi informasi, tidak selalu menaikkan skiil).

In [ ]:
corr = df_clean.corr(numeric_only=True)
plt.figure(figsize=(6, 4.5))
plt.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(corr)), corr.columns, rotation=90)
plt.yticks(range(len(corr)), corr.columns)
plt.colorbar(label="Korelasi Pearson")
plt.title("Matriks korelasi silang")
plt.tight_layout(); plt.show()
print(corr.round(2))

### Catatan format NetCDF/GRIB (Tujuan Bab 6 #2)

Notebook ini memakai data CSV (hasil download). Format grid **NetCDF** (`.nc`) dan **GRIB** dibahas
di teks §6.3: skrip `scripts/download_era5.py` unduh `.nc` (xarray/netCDF4) lalu konversi
ke `*_daily.csv` via `--process` — hasilnya file yang dipakai di sini (bila data nyata
tersedia). Untuk GRIB (mis. reanalisis tekanan/angin grid), panduan loading di teks.

### 3c. Membaca format NetCDF (Tujuan Bab 6 #2) - hands-on

Sel di bawah membuka berkas `.nc` **ERA5-Land** yang ter-commit di repo dengan xarray.
Catatan: hasil `scripts/download_era5.py` disimpan sebagai berkas yang diberi ekstensi `.nc`,
tetapi sebenarnya merupakan **arsip ZIP** berisi netCDF (pola CDS). Karena itu kode di bawah
mendeteksi `PK` (tanda ZIP) lalu mengekstrak membacanya secara in-memory - tepat sama dengan
pola `_open_era5` pada skrip. Ini menunjukkan dua hal: (1) membaca **NetCDF** dengan xarray,
dan (2) kewaspadaan terhadap format berkas yang tidak selalu seperti ekstensinya.


In [ ]:
import io, zipfile
import netCDF4

src = _BASE / "era5" / "jakarta" / "era5land_jakarta_2018.nc"
blob = src.read_bytes()

if blob[:2] == b"PK":                     # zip berisi netCDF (pola CDS)
    with zipfile.ZipFile(io.BytesIO(blob)) as zf:
        member = [m for m in zf.namelist() if m.endswith(".nc")][0]
        raw = zf.read(member)
else:                                     # netCDF murni
    raw = blob

# netCDF4 support memory I/O (pola _open_era5 di scripts/download_era5.py)
nc = netCDF4.Dataset("mem.nc", memory=raw)
print("Variabel:", list(nc.variables))
print("Dimensi:", {k: len(nc.dimensions[k]) for k in nc.dimensions})

# contoh agregasi: suhu permukaan t2m (Kelvin) -> Celsius rerata harian
t2m = nc.variables["t2m"][:]
print(f"Berhasil dibaca: {t2m.size:,} nilai t2m; rerata = {float(t2m.mean() - 273.15):.2f} C")
nc.close()


## 4. Feature Engineering

Fitur: deret tunda, musiman sinus, dan indeks iklim nyata (RMM/ONI dari `scripts/download_indices.py`).

In [4]:
from sklearn.preprocessing import StandardScaler

df_feat = df_clean.copy()
for lag in [1, 2, 3, 7, 14]:
    df_feat[f"hujan_t{lag}"] = df_feat["r_hujan"].shift(lag)
    for v in ("suhu", "angin", "kelembapan", "chirps_mm"):
        if v in df_feat:
            df_feat[f"{v}_t{lag}"] = df_feat[v].shift(lag)

df_feat["mus_sin"] = np.sin(2 * np.pi * df_feat.index.dayofyear / 365.25)
df_feat["mus_cos"] = np.cos(2 * np.pi * df_feat.index.dayofyear / 365.25)

# indeks iklim nyata: RMM harian (BoM) & ONI monthly (CPC) dari scripts/download_indices.py
_RAW = _buscar_raiz() / "manuscripts/ch-09-studi-kasus-curah-hujan-terbuka/data/raw"
if not (_RAW / "indeks_rmm.csv").exists() or not (_RAW / "indeks_oni.csv").exists():
    raise FileNotFoundError(
        "Indeks iklim (RMM/ONI) belum tersedia. Jalankan:\n"
        "  python scripts/download_indices.py"
    )
rmm = pd.read_csv(_RAW / "indeks_rmm.csv", parse_dates=["tanggal"]).set_index("tanggal")
rmm = rmm[~rmm.index.duplicated(keep="first")]
df_feat["rmm1"] = rmm["rmm1"].reindex(df_feat.index, method="ffill")
oni = pd.read_csv(_RAW / "indeks_oni.csv", parse_dates=["tanggal"]).set_index("tanggal")
oni = oni[~oni.index.duplicated(keep="first")]
df_feat["nino34"] = oni["anom"].reindex(df_feat.index, method="ffill")

feat_cols = [c for c in df_feat.columns if c != "r_hujan"]
print("Fitur:", feat_cols)

## 5. Housekeeping: buang baris awal (lag -> NaN) & split berbasis waktu

In [5]:
df_ml = df_feat.dropna().copy()
data = df_ml[feat_cols]
target = df_ml["r_hujan"]

n = len(data)
n_train = int(n * 0.7); n_val = int(n * 0.15)

scale = StandardScaler().fit(data.iloc[:n_train])
X_train = scale.transform(data.iloc[:n_train])
X_val = scale.transform(data.iloc[n_train:n_train+n_val])
X_test = scale.transform(data.iloc[n_train+n_val:])
y_train, y_val, y_test = target.iloc[:n_train], target.iloc[n_train:n_train+n_val], target.iloc[n_train+n_val:]

print("train", X_train.shape, "| val", X_val.shape, "| test", X_test.shape)
print("Rentang:", df_ml.index[0].date(), "->", df_ml.index[-1].date())

train (2035, 21) | val (436, 21) | test (437, 21)
Rentang: 2018-01-15 -> 2025-12-31


## 6. Latihan Mini

1. Ulangi dengan transformasi `log1p` pada target hujan; bandingkan distribusinya.
2. Buat fungsi pipeline reusable `make_dataset(csv_path) -> X, y, dates, scaler` untuk Bab 8–9.
3. Periksa efekt indeks nyata MJO/ENSO (RMM/ONI) pada model.
4. Terapkan *walk-forward* sederhana dan bandingkan MAE dengan split tunggal.